# Toy Model Validation and Tests

This notebook is the follow-up to the training notebook.

Goal: prove that the saved toy model actually works, not just that training ran.

What we validate:
1. Artifacts load correctly.
2. Data and tensor shapes are consistent.
3. Model produces valid probability distributions.
4. Baseline metrics are better than random chance.
5. Top-k predictions look sensible on known prompts.
6. Robustness checks on OOV and edge-case prompts.
7. Repeatability checks to ensure deterministic inference.
8. A compact pass/fail summary.

## 1) Imports and Artifact Paths

Set up libraries and point to files saved by the training notebook.

In [ ]:
import json
import math
import pickle
from pathlib import Path

import numpy as np

BASE_DIR = Path(".")
ARTIFACT_DIR = BASE_DIR / "artifacts"
VOCAB_PATH = ARTIFACT_DIR / "toy_vocab.json"
PARAMS_PATH = ARTIFACT_DIR / "toy_model.pkl"

print("Artifact directory:", ARTIFACT_DIR.resolve())
print("Vocab exists:", VOCAB_PATH.exists())
print("Params exists:", PARAMS_PATH.exists())

## 2) Load Artifacts and Reconstruct Mappings

Load vocabulary and parameters, then rebuild integer mappings so inference can run.

In [9]:
# Ensure required modules and paths exist in the current namespace; if not, create defaults.
try:
    json  # noqa: F401
except NameError:
    import json

try:
    pickle  # noqa: F401
except NameError:
    import pickle

try:
    VOCAB_PATH  # noqa: F401
    PARAMS_PATH  # noqa: F401
except NameError:
    from pathlib import Path

    BASE_DIR = Path(".")
    ARTIFACT_DIR = BASE_DIR / "artifacts"
    VOCAB_PATH = ARTIFACT_DIR / "toy_vocab.json"
    PARAMS_PATH = ARTIFACT_DIR / "toy_model.pkl"

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab_payload = json.load(f)

with open(PARAMS_PATH, "rb") as f:
    params = pickle.load(f)

word_to_id = vocab_payload["word_to_id"]
# JSON keys are strings, convert back to int keys
id_to_word = {int(k): v for k, v in vocab_payload["id_to_word"].items()}

vocab_size = len(word_to_id)
E = params["E"]
W = params["W"]
b = params["b"]
EMBED_DIM = E.shape[1]

print("Loaded vocab size:", vocab_size)
print("Parameter shapes: E", E.shape, "W", W.shape, "b", b.shape)

assert E.shape[0] == vocab_size
assert W.shape == (EMBED_DIM, vocab_size)
assert b.shape == (vocab_size,)
print("Shape checks passed.")

Loaded vocab size: 48
Parameter shapes: E (48, 16) W (16, 48) b (48,)
Shape checks passed.


## 3) Rebuild Toy Evaluation Data

Recreate the same tiny corpus and next-token examples used in training so we can evaluate model behavior consistently.

In [12]:
try:
    np  # noqa: F401
except NameError:
    import numpy as np

SEQ_LEN = 4

corpus = [
    "the model learns from text",
    "the tokenizer splits text into pieces",
    "byte pair encoding merges common pairs",
    "small corpora are good for debugging",
    "we train a tiny language model",
    "the residual stream tracks progress",
    "attention starts with token representations",
    "we test before we scale",
    "paper notes guide implementation choices",
    "weekly builds record what broke",
]


def tokenize_text(lines):
    tokens = []
    for line in lines:
        tokens.extend(line.lower().split())
    return tokens


def build_examples(token_ids, seq_len):
    X, y = [], []
    for i in range(len(token_ids) - seq_len):
        X.append(token_ids[i : i + seq_len])
        y.append(token_ids[i + seq_len])
    return np.array(X, dtype=np.int64), np.array(y, dtype=np.int64)

all_tokens = tokenize_text(corpus)
encoded_tokens = [word_to_id[w] for w in all_tokens if w in word_to_id]
X, y = build_examples(encoded_tokens, SEQ_LEN)

print("Tokens:", len(all_tokens), "| Examples:", len(X))
print("Example X[0]:", X[0], "-> y[0]:", y[0])

Tokens: 54 | Examples: 50
Example X[0]: [38 21 19 13] -> y[0]: 37


## 4) Inference Functions

Define forward pass and helper utilities used for model quality tests.

In [13]:
def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=1, keepdims=True)


def forward(batch_X, model_params):
    embedded = model_params["E"][batch_X]
    hidden = embedded.mean(axis=1)
    logits = hidden @ model_params["W"] + model_params["b"]
    probs = softmax(logits)
    return logits, probs


def cross_entropy(probs, targets):
    return -np.mean(np.log(probs[np.arange(len(targets)), targets] + 1e-12))


def top_k_accuracy(probs, targets, k=1):
    topk = np.argsort(probs, axis=1)[:, -k:]
    hits = [(targets[i] in topk[i]) for i in range(len(targets))]
    return float(np.mean(hits))


def predict_next(prompt_words, model_params, seq_len=4, top_k=5):
    prompt_words = [w.lower() for w in prompt_words]
    if len(prompt_words) < seq_len:
        pad = ["the"] * (seq_len - len(prompt_words))
        prompt_words = pad + prompt_words
    else:
        prompt_words = prompt_words[-seq_len:]

    unk_id = word_to_id.get("the", 0)
    ids = np.array([[word_to_id.get(w, unk_id) for w in prompt_words]], dtype=np.int64)
    _, probs = forward(ids, model_params)
    p = probs[0]
    top_ids = np.argsort(p)[-top_k:][::-1]
    return [(id_to_word[i], float(p[i])) for i in top_ids], ids

## 5) Quantitative Metrics (Proof by Numbers)

We compute cross-entropy, perplexity, top-1 accuracy, and top-3 accuracy.

We also compare top-1 accuracy against random chance (1 / vocab_size).

In [15]:
try:
	math
except NameError:
	import math

_, probs = forward(X, params)
loss = cross_entropy(probs, y)
perplexity = float(math.exp(loss))
acc_top1 = top_k_accuracy(probs, y, k=1)
acc_top3 = top_k_accuracy(probs, y, k=3)

random_baseline_top1 = 1.0 / vocab_size
improvement_vs_random = acc_top1 - random_baseline_top1

print(f"Cross-entropy loss: {loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")
print(f"Top-1 accuracy: {acc_top1:.4f}")
print(f"Top-3 accuracy: {acc_top3:.4f}")
print(f"Random baseline Top-1: {random_baseline_top1:.4f}")
print(f"Improvement over random: {improvement_vs_random:+.4f}")

Cross-entropy loss: 3.6175
Perplexity: 37.2437
Top-1 accuracy: 0.1000
Top-3 accuracy: 0.1600
Random baseline Top-1: 0.0208
Improvement over random: +0.0792


## 6) Qualitative Predictions (Proof by Behavior)

Check top predictions for prompts that should feel familiar from the corpus.

In [16]:
prompts = [
    ["the", "tokenizer", "splits", "text"],
    ["we", "test", "before", "we"],
    ["paper", "notes", "guide", "implementation"],
    ["the", "model", "learns", "from"],
]

for p in prompts:
    preds, ids = predict_next(p, params, seq_len=SEQ_LEN, top_k=5)
    print("Prompt:", " ".join(p))
    print("Encoded prompt IDs:", ids.tolist()[0])
    print("Top-5:", preds)
    print("-")

Prompt: the tokenizer splits text
Encoded prompt IDs: [38, 41, 33, 37]
Top-5: [('text', 0.06149976965919055), ('the', 0.05076628742370995), ('we', 0.047062944312817524), ('stream', 0.028938793465982077), ('splits', 0.026923499870275532)]
-
Prompt: we test before we
Encoded prompt IDs: [44, 36, 3, 44]
Top-5: [('we', 0.08777759307045363), ('the', 0.030110054805103932), ('scale', 0.028102965128555098), ('tiny', 0.027856918945274547), ('text', 0.027294631989237716)]
-
Prompt: paper notes guide implementation
Encoded prompt IDs: [25, 22, 15, 16]
Top-5: [('we', 0.05890665486601894), ('the', 0.03352151141307998), ('text', 0.03153447222238915), ('implementation', 0.02546746930781127), ('weekly', 0.02346105783043188)]
-
Prompt: the model learns from
Encoded prompt IDs: [38, 21, 19, 13]
Top-5: [('text', 0.06584332562062761), ('the', 0.055409259645053825), ('we', 0.04597188294369771), ('stream', 0.030258896988392927), ('splits', 0.027671234598261003)]
-


## 7) Robustness and Safety Checks

Test edge cases:
- unknown words (OOV)
- very short prompts
- deterministic repeat calls

In [17]:
# OOV prompt should not crash
oov_prompt = ["quantum", "llamas", "debug", "tokenizers"]
oov_preds_1, _ = predict_next(oov_prompt, params, seq_len=SEQ_LEN, top_k=3)
oov_preds_2, _ = predict_next(oov_prompt, params, seq_len=SEQ_LEN, top_k=3)

# Very short prompt should auto-pad and still work
short_prompt = ["the"]
short_preds, short_ids = predict_next(short_prompt, params, seq_len=SEQ_LEN, top_k=3)

# Determinism for same prompt and same model
deterministic_ok = oov_preds_1 == oov_preds_2

print("OOV predictions:", oov_preds_1)
print("Short prompt encoded IDs:", short_ids.tolist()[0])
print("Short prompt predictions:", short_preds)
print("Deterministic inference:", deterministic_ok)

assert deterministic_ok, "Expected deterministic inference for same input"
assert len(short_ids.tolist()[0]) == SEQ_LEN, "Expected left-padding to SEQ_LEN"
print("Robustness checks passed.")

OOV predictions: [('text', 0.08730306745633801), ('the', 0.06240012008379526), ('we', 0.04060884001293568)]
Short prompt encoded IDs: [38, 38, 38, 38]
Short prompt predictions: [('text', 0.08730306745633801), ('the', 0.06240012008379526), ('we', 0.04060884001293568)]
Deterministic inference: True
Robustness checks passed.


## 8) Final Pass/Fail Report

This section gives a clear conclusion on whether the toy model is working well enough for the next phase.

In [18]:
criteria = {
    "artifacts_loaded": VOCAB_PATH.exists() and PARAMS_PATH.exists(),
    "shape_checks": (E.shape[0] == vocab_size and W.shape == (EMBED_DIM, vocab_size) and b.shape == (vocab_size,)),
    "probabilities_valid": np.allclose(probs.sum(axis=1), 1.0, atol=1e-6),
    "better_than_random": acc_top1 > random_baseline_top1,
    "top3_ge_top1": acc_top3 >= acc_top1,
    "deterministic_inference": deterministic_ok,
}

passed = sum(bool(v) for v in criteria.values())
total = len(criteria)

print("Validation report")
print("=" * 40)
for k, v in criteria.items():
    print(f"{k:22s}:", "PASS" if v else "FAIL")

print("=" * 40)
print(f"Passed {passed}/{total} checks")

if passed == total:
    print("Conclusion: The toy model is working for this scope. You can move to the BPE-specific phase.")
else:
    print("Conclusion: Fix failing checks before moving on.")

Validation report
artifacts_loaded      : PASS
shape_checks          : PASS
probabilities_valid   : PASS
better_than_random    : PASS
top3_ge_top1          : PASS
deterministic_inference: PASS
Passed 6/6 checks
Conclusion: The toy model is working for this scope. You can move to the BPE-specific phase.
